# 05 BentoML Integration
This notebook demonstrates how to save the trained pneumonia classifier to the local BentoML model store and test its loading/prediction flow.

In [1]:
import torch
import bentoml
from PIL import Image
import numpy as np
import io
import os
from torchvision import transforms

from pneumonia_classifier.ml.model.arch import Net

# Constants
BENTOML_MODEL_NAME = "pneumonia_classifier_model"
TRAIN_TRANSFORMS_KEY = "pneumonia_classifier_train_transforms"

# Load the state_dict (assuming it was saved in 03_model_training.ipynb)
device = torch.device("cpu")
model = Net().to(device)
try:
    model.load_state_dict(torch.load('model_v1.pt', map_location=device))
    print("Model loaded successfully.")
except FileNotFoundError:
    print("Error: model_v1.pt not found. Please run 03_model_training.ipynb first.")

model.eval()

# Define transforms for inference
inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

Model loaded successfully.


## Save Model to BentoML

In [ ]:
# Save the model to local BentoML model store
try:
    bento_model = bentoml.pytorch.save_model(
        name=BENTOML_MODEL_NAME,
        model=model,
        custom_objects={
            TRAIN_TRANSFORMS_KEY: inference_transform
        }
    )
    print(f"Model saved to BentoML: {bento_model}")
except Exception as e:
    print(f"Error saving model to BentoML: {e}")

## Verify and Test Prediction

In [ ]:
try:
    loaded_model = bentoml.pytorch.get(BENTOML_MODEL_NAME)
    runner = loaded_model.to_runner()
    runner.init_local()
    print("BentoML runner initialized locally.")

    # Test with a sample image
    sample_path = r'j:\Users\ayush\Desktop\code\pneumonia_classifier\artifacts\02_12_2025_08_52_04\data_ingestion\data\data\test\PNEUMONIA\person100_bacteria_475.jpeg'
    if os.path.exists(sample_path):
        img = Image.open(sample_path).convert('RGB')
        transformed_img = inference_transform(img).unsqueeze(0)
        
        with torch.no_grad():
            # Use the runner for prediction
            output = runner.run(transformed_img)
            prediction = torch.argmax(output, dim=1).item()
            label = "PNEUMONIA" if prediction == 1 else "NORMAL"
            print(f"Prediction: {label} ({prediction})")
    else:
        print("Sample image not found for testing.")
except Exception as e:
    print(f"Error during BentoML inference test: {e}")